# Data Preprocessing Pipelines — AI Lab Instructor Notebook

*Deep Learning · Medium*

Make preprocessing *correct and repeatable* using `Pipeline` + `ColumnTransformer`, while avoiding the most common leakage bugs.

**Outcome.** Students can:
- build end-to-end sklearn pipelines for mixed-type data,
- handle missing values, scaling, and one-hot encoding correctly,
- use CV without leakage,
- debug common pipeline mistakes (wrong dtypes, unseen categories, fit on full data).

6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Data Preprocessing Pipelines — Student Lab (Customer Churn)

Focus: build leakage-safe preprocessing with `ColumnTransformer` + `Pipeline`.

## Section 0 — Load churn dataset (with fallback)
Expected path: `data/churn/churn.csv`

In [ ]:
$$10

**Why this works.** # Data Preprocessing Pipelines — Instructor (Customer Churn)

## Task 2: Create y in {0,1}
*Section: Robust target parsing*

## Section 1 — Robust target parsing

### Task 1.1: Create y in {0,1}

In [ ]:
df2 = df.copy()
target_col = 'Churn' if 'Churn' in df2.columns else ('churn' if 'churn' in df2.columns else None)
if target_col is None:
    raise ValueError('Target column not found: expected Churn or churn')
y_raw = df2[target_col]
if y_raw.dtype == object:
    y = y_raw.astype(str).str.lower().map({'yes': 1, 'no': 0})
else:
    y = y_raw.astype(int)
if y.isna().any():
    raise ValueError('Target contains unexpected values; expected Yes/No or 0/1')
X = df2.drop(columns=[target_col])
print('base_rate', float(np.mean(y)))

In [ ]:
# Checks
check('target_found', target_col is not None)
check('no_nan_y', y.notna().all())

## Task 3: Identify numeric vs categorical columns
*Section: Build ColumnTransformer*

## Section 2 — Build ColumnTransformer

### Task 2.1: Identify numeric vs categorical columns
### Task 2.2: Create preprocess with two branches

Numeric: impute median + scale
Categorical: impute most_frequent + one-hot (handle_unknown='ignore')

In [ ]:
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]
preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ]
)

## Task 4: Create stratified train/holdout split
*Section: Stratified holdout split*

## Section 3 — Stratified holdout split

### Task 3.1
Create an explicit holdout split:
- train: 80%
- holdout: 20%
with stratification on `y`.

Keep holdout untouched until the final check.

**Hints**
- use train_test_split(..., test_size=0.2, stratify=y.values, random_state=0)
- verify both classes are present in train and holdout

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y.values, test_size=0.2, stratify=y.values, random_state=0
)

check('split_sizes', len(X_train) + len(X_test) == len(X))
check('both_classes_present', len(np.unique(y_train)) == 2 and len(np.unique(y_test)) == 2)
print('train_pos_rate', float(np.mean(y_train)), 'holdout_pos_rate', float(np.mean(y_test)))

In [ ]:
# Checks
check('split_sizes', len(X_train) + len(X_test) == len(X))
check('both_classes_present', len(np.unique(y_train)) == 2 and len(np.unique(y_test)) == 2)

**Why this works.** preprocessing is fit within each fold, preventing leakage.

## Task 5: Run StratifiedKFold CV on X_train only
*Section: CV on training split only*

## Section 4 — CV on training split only

### Task 4.1
Run 5-fold `StratifiedKFold` CV on `X_train, y_train` using a `Pipeline(preprocess -> model)`.

### Task 4.2
(Leakage demo) show the wrong way by fitting preprocess on all `X_train` before CV.

**Hints**
- `cross_validate(pipe, X_train, y_train, cv=cv, ...)` keeps preprocessing inside folds
- for leakage demo, try preprocess.fit_transform(X_train) before CV

In [ ]:
pipe = Pipeline(steps=[('preprocess', preprocess), ('model', LogisticRegression(max_iter=5000))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring={'roc_auc': 'roc_auc'}, return_train_score=False)
right = float(np.mean(scores['test_roc_auc']))
print('train_cv_roc_auc', right)

# Wrong way (leakage inside CV folds on training split):
X_train_all = preprocess.fit_transform(X_train)
leaky_scores = cross_val_score(LogisticRegression(max_iter=5000), X_train_all, y_train, cv=cv, scoring='roc_auc')
wrong = float(np.mean(leaky_scores))
print('train_cv_leaky_roc_auc', wrong)

check('cv_auc_range', 0.0 <= right <= 1.0)
check('cv_used_train_only', len(X_train) < len(X))

In [ ]:
# Checks
check('cv_auc_range', 0.0 <= right <= 1.0)
check('cv_used_train_only', len(X_train) < len(X))

**Why this works.** fitting preprocess on full data leaks fold information and inflates scores.

## Task 6: Fit on X_train and evaluate X_test
*Section: Final holdout evaluation*

## Section 5 — Final holdout evaluation

### Task 5.1
Fit the pipeline on the full training split (`X_train, y_train`) and evaluate ROC-AUC on untouched holdout (`X_test, y_test`).

This is your final unbiased estimate after model development on training data.

**Hints**
- fit only on training split
- use predict_proba on holdout and roc_auc_score

In [ ]:
pipe_holdout = Pipeline(steps=[('preprocess', preprocess), ('model', LogisticRegression(max_iter=5000))])
pipe_holdout.fit(X_train, y_train)
p_test = pipe_holdout.predict_proba(X_test)[:, 1]
holdout_auc = float(roc_auc_score(y_test, p_test))
print('holdout_roc_auc', holdout_auc)

check('holdout_auc_range', 0.0 <= holdout_auc <= 1.0)
check('holdout_size_nonzero', len(X_test) > 0)

In [ ]:
# Checks
check('holdout_auc_range', 0.0 <= holdout_auc <= 1.0)
check('holdout_size_nonzero', len(X_test) > 0)